In [0]:
# src/notebooks/01_data_profiling.py
# Branch: feature/data-profiling
# PURPOSE: Profile ALL 5 dataset files before chunking
from pyspark.sql.functions import (
    col, count, when, isnull, isnan, countDistinct, min, max, avg, round as spark_round
)

CATALOG = "vstone_catalog"
LANDING_PATH = f"/Volumes/{CATALOG}/raw/landing"

# ===== FILE MANIFEST =====
FILES = {
    "1_main": {"path": f"{LANDING_PATH}/1_main.csv", "format": "csv", "sep": ","},
    "catalogs": {"path": f"{LANDING_PATH}/catalogs.csv", "format": "csv", "sep": ","},
    "geolocation": {"path": f"{LANDING_PATH}/final_geografic.csv", "format": "csv", "sep": ","},
    "text": {"path": f"{LANDING_PATH}/1_text.csv", "format": "csv", "sep": ","},
    "photos": {"path": f"{LANDING_PATH}/1_photo.csv", "format": "csv", "sep": ","},
}

def profile_file(name, path, fmt, sep=","):
    print(f"\n{'='*70}")
    print(f"FILE: {name.upper()}")
    print(f"{'='*70}")
    df = (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("sep", sep)
        .option("encoding", "UTF-8")
        .format(fmt)
        .load(path)
    )
    total_rows = df.count()
    
    # --- ADDED: DUPLICATE COUNT LOGIC ---
    distinct_rows = df.distinct().count()
    duplicate_count = total_rows - distinct_rows
    
    print(f"Total Rows : {total_rows:,}")
    print(f"Total Columns: {len(df.columns)}")
    print(f"Duplicate Records: {duplicate_count:,}") #
    print(f"Columns : {df.columns}")
    df.printSchema()
    
    # --- ADDED: NULL ANALYSIS WITH PERCENTAGE ---
    print(f"\n--- NULL ANALYSIS (COUNT & PERCENTAGE) ---")
    null_stats_df = df.select([
        count(when(isnull(c), c)).alias(f"{c}_null_count") for c in df.columns
    ] + [
        spark_round((count(when(isnull(c), c)) / total_rows) * 100, 2).alias(f"{c}_null_percentage") 
        for c in df.columns
    ])
    display(null_stats_df)
    
    # Sample data
    print(f"\n--- SAMPLE (3 rows) ---")
    display(df.limit(3))
    return df, total_rows

# Profile all files and store results
profiled = {}
for name, cfg in FILES.items():
    df, rows = profile_file(name, cfg["path"], cfg["format"], cfg.get("sep", ","))
    profiled[name] = {"df": df, "rows": rows}

# Key stats for 1_main.csv (primary transaction file)
print("\n=== 1_MAIN.CSV KEY STATS ===")
df_main = profiled["1_main"]["df"]
display(
    df_main.select(
        min("year").alias("min_year"),
        max("year").alias("max_year"),
        min("cost").alias("min_price_rub"),
        max("cost").alias("max_price_rub"),
        avg("cost").alias("avg_price_rub"),
        countDistinct("marka").alias("unique_brands"),
        countDistinct("model").alias("unique_models"),
        countDistinct("place").alias("unique_cities"),
        min("date").alias("earliest_listing"),
        max("date").alias("latest_listing")
    )
)

print("\n=== TOP 10 BRANDS ===")
display(df_main.groupBy("marka").count().orderBy("count", ascending=False).limit(10))

print("\n=== FUEL TYPE DISTRIBUTION ===")
display(df_main.groupBy("engine").count().orderBy("count", ascending=False))

print("\n=== LISTINGS BY YEAR ===")
display(df_main.groupBy("year").count().orderBy("year").limit(20))

In [0]:
# Sirf Listing ID ke level par duplicate check karne ke liye code
id_duplicates = profiled["photos"]["df"].count() - profiled["photos"]["df"].select("id").distinct().count()
print(f"Duplicate Listing IDs in Photos: {id_duplicates}")